# DeepMIR HW1: reproducing the test predictions

This notebook reproduces my submission `b12901045.json` from the trained checkpoints.

| | Task A: release decade (`dataset_A`) | Task B: release market (`dataset_B`) |
|---|---|---|
| Model | Short-Chunk CNN on log-mel spectrograms | Dense head on frozen Whisper + MERT features |
| Input | Raw audio of the mix | Pre-extracted features (downloaded below) |
| Script | `inference.py` | `inference_features.py` |
| Output | `predictions_A.json` | `predictions_B.json` |

Run the cells from top to bottom. A GPU runtime is recommended (Colab: *Runtime → Change runtime type → T4 GPU*), but both scripts also run on CPU.

## 0. Get the code

In [ ]:
import os

# Skip cloning if the notebook is already running inside the hw1 folder
if not os.path.exists("inference.py"):
    !git clone https://github.com/wy11ing/DeepMIR_2026fall.git
    %cd DeepMIR_2026fall/hw1

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt gdown

## 2. Download the data

- `dataset_A`: the audio clips for Task A (the CNN runs on the raw audio).
- `dataset_B`: the manifest for Task B (test sample ids).
- `features.zip`: the Whisper / MERT features of every `dataset_B` clip that the Task B model reads. Extracting them from scratch needs Demucs source separation and both large models (see the README), so they are provided pre-extracted.

In [ ]:
!gdown 17FwHQOWcxqHE4TzMgZKDkk6nZW04S2-u -O dataset_A.zip   # dataset_A.zip (1.8 GB)
!unzip -q -o dataset_A.zip

!gdown 1y2QZWV2QESYGGoMZDXy4zt9ezbtsOhK_ -O dataset_B.zip   # dataset_B.zip (1.4 GB)
!unzip -q -o dataset_B.zip

!gdown 1flinRduwVs4JIkjyPpE-T5dc-NI52TFS -O features.zip    # pre-extracted features (0.5 GB)
!unzip -q -o features.zip "features/market/*"

## 3. Download the checkpoints

In [ ]:
# TODO: fill in the Google Drive file ids of the two checkpoints
TASK_A_CKPT_ID = "1shbN-Z3L6ePksEoVjjATCW-fvEZwC3A4"  # Short-Chunk CNN (Task A)
TASK_B_CKPT_ID = "1c8RLaf_d4V-mHgRMv_aE7fTthia-oozW"  # dense head on Whisper + MERT features (Task B)

!mkdir -p checkpoints
!gdown {TASK_A_CKPT_ID} -O checkpoints/taskA_shortchunk_cnn.pt
!gdown {TASK_B_CKPT_ID} -O checkpoints/taskB_dense_head.pt

## 4. Task A inference: Short-Chunk CNN

Each 30 s clip is resampled to 24 kHz and converted to a 128-band log-mel spectrogram. The CNN (7 conv blocks with 64 to 256 channels, global max pooling, then a dense head) scores 10 evenly spaced, overlapping 4 s crops of the clip. The logits are averaged over the crops, and the top 3 decades are kept.

The crop settings and network width are read from the checkpoint, so no flags are needed.

In [ ]:
!python inference.py --ckpt checkpoints/taskA_shortchunk_cnn.pt --output predictions_A.json --num_workers 2

## 5. Task B inference: dense head on frozen Whisper + MERT features

Each clip is described by three pre-extracted inputs:

- **Whisper large-v3** encoder on the Demucs **vocal stem**: all 33 hidden layers, each mean + std pooled over vocal-active frames.
- **Whisper language-ID** log-probabilities (what language is sung).
- **MERT-v1-330M** on the **mix**: all 25 hidden layers, mean + std pooled.

A learned softmax weighting mixes each model's layers into one vector. The vectors are concatenated and passed to the same dense head as the CNN (Linear → BatchNorm → ReLU → Dropout → Linear). The checkpoint records which feature files it was trained on, so no flags are needed.

In [ ]:
!python inference_features.py --ckpt checkpoints/taskB_dense_head.pt --output predictions_B.json

## 6. Merge into the submission file

In [ ]:
import json

with open("predictions_A.json") as f:
    submission = json.load(f)  # {"dataset_A": {...}}
with open("predictions_B.json") as f:
    submission.update(json.load(f))  # {"dataset_B": {...}}

with open("b12901045.json", "w") as f:
    json.dump(submission, f, indent=2)

for key, preds in submission.items():
    print(f"{key}: {len(preds)} test clips, e.g. {next(iter(preds.items()))}")